# Phase 9: few-step baselines before training a lightning LoRA

**Version 1** is the unhealed 30-block student (blocks 5 and 20 replaced by attention bridges; Phase 7 and 8). It
samples in 40 steps without classifier-free guidance, so cutting to 8 steps is about five times faster and to 4 steps
about ten times. Our own 8-step and 4-step lightning LoRAs are trained in Phase 10 (`phase10-lightning-lora.ipynb`).

This phase sets the reference points, all on the **intact teacher** with the same prompts and seeds as every earlier
grid (5 text prompts with 4 seeds, 25 general prompts):

| Variant | What it shows |
|---|---|
| teacher, 8 steps, no LoRA | how much quality few steps lose without distillation |
| teacher, 4 steps, no LoRA | the same at 4 steps |
| teacher + LightX2V Lightning LoRA, 4 steps | what a published 4-step LoRA (DMD, rank 16 on q, k, v, out) achieves, for comparison only; it is not reused |

The LightX2V LoRA is pinned to revision `6b0a1176` and its file hash is checked; it is merged into the weights
(W + up·down, alpha/rank = 1) and run with its model card's schedule (sigmas 1, 0.75, 0.5, 0.25; no terminal stretch).

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
assert (ROOT / 'PLAN.md').exists(), 'Run this notebook from the repository root.'
sys.path.insert(0, str(ROOT / 'tools'))
for stream in (sys.stdout, sys.stderr):      # Modal prints non-ASCII progress symbols
    if hasattr(stream, 'reconfigure'):
        stream.reconfigure(encoding='utf-8')
ALLOW_GPU = False          # set True only to launch a step whose saved result is missing
SOURCES = {}               # worker sources, embedded per step exactly as executed


def load_summary(run):
    path = ROOT / run / 'summary.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def reuse_or_launch(run, launch):
    summary = load_summary(run)
    if summary is not None:
        print('Saved result:', run, '| status', summary.get('status'))
        return summary
    if (ROOT / run).parent.joinpath('aborted.json').exists():
        print('Run stopped by request; nothing to reuse.')
        return None
    if not ALLOW_GPU:
        raise RuntimeError(f'No saved result for {run}; set ALLOW_GPU = True to launch it on Modal.')
    launch()
    return load_summary(run)


def pct(value, digits=2):
    return f'{100 * value:.{digits}f}%'


def velocity_row(label, block):
    o = block['overall']['teacher_relative_l2']
    p = {k: block['panels'][k]['teacher_relative_l2']['mean'] for k in block['panels']}
    s = {k: block['stages'][k]['teacher_relative_l2']['mean'] for k in block['stages']}
    return (f"| {label} | {pct(o['mean'])} | {pct(p['gate10'])} | {pct(p['wider50'])} | "
            f"{pct(s['early'], 1)} / {pct(s['middle'], 1)} / {pct(s['late'], 1)} | {pct(o['p90'], 1)} | {pct(o['maximum'], 1)} |")


def velocity_table(rows):
    head = '| Model | Overall | Check (10) | Wider (50) | Early / middle / late | p90 | Worst |\n|---|---|---|---|---|---|---|\n'
    display(Markdown(head + '\n'.join(rows)))


def show_variants(summary):
    rows = [velocity_row(f"{name} (step {v['fit']['selected_step']}, {v['parameters'] / 1e6:.1f}M params)", v)
            for name, v in summary['variants'].items()]
    velocity_table(rows)


def show_images(folder, names):
    for name in names:
        display(Image(filename=str(ROOT / folder / name), width=1000))

## 1. Few-step baselines on the teacher

Quantitative: render time per image. Qualitative: the four-seed text grids and the 25 general prompts, beside the
teacher's 40-step images.

In [ ]:
# Few-step baselines: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['lightning_baseline_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase9/lightning_baseline') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [v['name'] for v in request['variants']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant, spec):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=spec['steps'], sigmas=spec.get('sigmas'), true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=spec['use_kv_cache'], output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        import urllib.request
        from diffusers import FlowMatchEulerDiscreteScheduler
        default_scheduler = pipe.scheduler
        outputs = {v: {} for v in variants}
        merged = False
        for spec in request['variants']:
            if spec.get('lora') and not merged:
                # Download the pinned LoRA, check its hash, and merge it into the weights: W <- W + (alpha / rank) * up @ down.
                lora = request['lora']
                path = Path('/tmp') / lora['filename']
                if not path.exists():
                    urllib.request.urlretrieve(lora['url'], path)
                if digest(path) != lora['sha256']:
                    raise RuntimeError('LoRA file differs.')
                from safetensors.torch import load_file
                weights = load_file(str(path))
                scale = lora['alpha'] / lora['rank']
                modules = dict(model.named_modules())
                count = 0
                with torch.no_grad():
                    for name in sorted(k for k in weights if k.endswith('.lora.down.weight')):
                        target = name[len('transformer.'):-len('.lora.down.weight')]
                        down, up = weights[name].to('cuda', torch.float32), weights[name.replace('.down.', '.up.')].to('cuda', torch.float32)
                        layer = modules[target]
                        layer.weight.copy_((layer.weight.float() + scale * (up @ down)).to(layer.weight.dtype))
                        count += 1
                if count != 32 * 4:
                    raise RuntimeError('Merged LoRA count differs.')
                report['lora_merged'] = count
                merged = True
            if spec.get('lora') is not True and merged:
                raise RuntimeError('Variants without the LoRA must come first.')
            if spec.get('scheduler_overrides'):
                pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(default_scheduler.config, **spec['scheduler_overrides'])
            else:
                pipe.scheduler = default_scheduler
            model.eval()
            for row in request['rows']:
                outputs[spec['name']][row['source_id']] = generate(row, spec['name'], spec)
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                              for r in text_rows for k in (1, 2, 3)]
    rows += [r for r in base['rows'] if r['subdimension'] not in ('text_rendering', 'posters')]
    revision = '6b0a1176327c01522a44276c5da67331a025b6fc'
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase9-lightning-baseline-v1', rows=rows, worker_source=SOURCES['lightning_baseline_worker.py'],
        execution_account='new', candidate_only=True,
        lora=dict(repo='lightx2v/Qwen-Image-2.1-Lightning', revision=revision, filename='pytorch_lora_weights.safetensors',
                  url=f'https://huggingface.co/lightx2v/Qwen-Image-2.1-Lightning/resolve/{revision}/pytorch_lora_weights.safetensors',
                  sha256='53c69320d4163da51032f90275074028cb098556039c72b45f8dce9f0f9e34a3', rank=16, alpha=16),
        variants=[dict(name='teacher_8naive', steps=8, use_kv_cache=True),
                  dict(name='teacher_4naive', steps=4, use_kv_cache=True),
                  # The model card's recipe: four fixed sigmas, dynamic exponential shift, no terminal stretch, no KV cache.
                  dict(name='teacher_lightx2v4', steps=4, sigmas=[1.0, 0.75, 0.5, 0.25], use_kv_cache=False, lora=True,
                       scheduler_overrides=dict(num_train_timesteps=1000, use_dynamic_shifting=True, time_shift_type='exponential',
                                                base_image_seq_len=256, max_image_seq_len=8192, base_shift=0.5, max_shift=0.9,
                                                shift_terminal=None))],
        scope='Few-step baselines on the intact teacher: naive 8 and 4 steps, and the LightX2V 4-step LoRA (reference only)')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase9/lightning_baseline' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase9/lightning_baseline/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase9-lightning-baseline')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_baseline_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'lora_merged')}), flush=True)
    return request_key

def build_baseline_sheets(run):
    """Text prompts: rows teacher 40 steps, naive 8, naive 4, LightX2V 4, seeds 0-3. General prompts: the same four as columns."""
    import hashlib
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    store = ROOT / '.archive/phase9_lightning_baseline' / key
    store.mkdir(parents=True, exist_ok=True)
    for artifact in json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))['artifacts']:
        path = store / artifact['name']
        if not path.exists():
            path.write_bytes(b''.join(volume.read_file(f"/phase9/lightning_baseline/{key}/{artifact['name']}")))
        if hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            raise RuntimeError('Image hash mismatch: ' + artifact['name'])
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 21)
    out = ROOT / run / 'sheets'
    out.mkdir(exist_ok=True)
    titles = ['Teacher, 40 steps', 'Teacher, 8 steps (no LoRA)', 'Teacher, 4 steps (no LoRA)', 'Teacher + LightX2V, 4 steps']
    suffixes = ['teacher_8naive', 'teacher_4naive', 'teacher_lightx2v4']
    tile, label = 300, 330
    for index, row in enumerate(first['rows']):
        sid = row['source_id']
        name = lambda k, suffix: f'{sid}-{suffix}.png' if k == 0 else f'{sid}-s{k}-{suffix}.png'
        if row['subdimension'] in ('text_rendering', 'posters'):
            locate = [lambda k: first_store / row['teacher']['name'] if k == 0 else reference / name(k, 'teacher')]
            locate += [lambda k, s=s: store / name(k, s) for s in suffixes]
            canvas = PILImage.new('RGB', (label + 4 * tile, 80 + 4 * tile), 'white')
            draw = ImageDraw.Draw(canvas)
            draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
            for column in range(4):
                draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
            for line, title in enumerate(titles):
                y = 80 + line * tile
                draw.text((10, y + tile // 2 - 12), title, font=big, fill='black')
                for column in range(4):
                    with PILImage.open(locate[line](column)) as image:
                        canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        else:
            paths = [first_store / row['teacher']['name']] + [store / name(0, s) for s in suffixes]
            side = 400
            canvas = PILImage.new('RGB', (4 * side, 90 + side), 'white')
            draw = ImageDraw.Draw(canvas)
            draw.text((10, 8), textwrap.shorten(f'P{index + 1:02d}  ' + row['prompt'], 170), font=font, fill='black')
            for column, (title, path) in enumerate(zip(titles, paths)):
                draw.text((column * side + 10, 50), title, font=big, fill='black')
                with PILImage.open(path) as image:
                    canvas.paste(image.convert('RGB').resize((side, side), PILImage.Resampling.LANCZOS), (column * side, 90))
        canvas.save(out / f'P{index + 1:02d}.jpg', quality=88)
    return out

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase9/lightning_baseline/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    import statistics
    for variant in summary['calls']:
        seconds = [r[variant]['seconds'] for r in summary['rows']]
        print(variant, 'median seconds per image', round(statistics.median(seconds), 2))
    sheets = build_baseline_sheets(RUN)
    show_images(str(sheets.relative_to(ROOT)), sorted(p.name for p in sheets.glob('*.jpg')))

### What the baseline results mean

| Variant | Exact signs (of 12) | Distance to the teacher's 40-step image (PSNR, median) | Seconds per image |
|---|---|---|---|
| teacher, 40 steps | 11 | (reference) | about 7 |
| teacher, 8 steps, no LoRA | 0: letters faint or broken, images smeared | 15.1 dB | 1.53 |
| teacher, 4 steps, no LoRA | 0: no usable image | 14.1 dB | 0.86 |
| teacher + LightX2V, 4 steps | **9**: Rusty Anchor 1 of 4 ("Acchor" three times), Emma 4, Motel 4 | **11.8 dB** | 0.87 |

1. **Without distillation, few steps are unusable**: 8 steps already smears structure and loses letters; 4 steps gives
   no image. Distillation is required.
2. **A professionally distilled 4-step LoRA loses spelling too**: LightX2V draws clean, sharp images and spells
   Emma and the motel sign, but writes "Acchor" in three of four seeds (teacher 11 of 12). Few-step sampling itself
   costs letters, so our training data is text-heavy (450 of 500 prompts).
3. **Distance to the teacher is not a quality measure for every method.** LightX2V has the cleanest images and the
   *lowest* PSNR: its distillation (DMD) matches the teacher's image distribution, not the teacher's image for the
   same noise, so it composes differently (P05: one man and a dog, posed differently). Our trajectory distillation
   aims at the teacher's image for the same noise, so PSNR is fair between naive sampling and our LoRAs, while
   spelling and the images compare all of them.

## 2. Speed and memory: teacher against version 1

What removing two blocks buys. Batch 1 at 1024x1024 on one H100, KV cache on, 5 general prompts x 3 repeats after a
warm-up, for 40, 8 and 4 steps. Each component is timed separately (text encoder, transformer, VAE decode), with
peak GPU memory and weight sizes. A fused LoRA does not change speed, so the 8- and 4-step rows also hold for the
distilled models.

In [ ]:
# Speed and memory benchmark: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True.
SOURCES['benchmark_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase9/benchmark') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'{m}_{s}' for m in ('teacher', 'v1') for s in request['steps']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant, spec):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=spec['steps'], sigmas=spec.get('sigmas'), true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=spec['use_kv_cache'], output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # Speed and memory: the same pipeline, prompts and noise for every variant; each component timed separately.
        import statistics
        outputs = {v: {} for v in variants}
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        timing = {}

        def timed_module(module, name):
            def before(*_):
                torch.cuda.synchronize()
                timing.setdefault('_start', {})[name] = time.perf_counter()
            def after(*_):
                torch.cuda.synchronize()
                timing.setdefault(name, []).append(time.perf_counter() - timing['_start'][name])
            module.register_forward_pre_hook(before)
            module.register_forward_hook(after)

        timed_module(pipe.transformer, 'transformer')
        timed_module(pipe.text_encoder, 'text_encoder')
        original_decode = pipe.vae.decode

        def decode(*args, **kwargs):
            torch.cuda.synchronize()
            started = time.perf_counter()
            result = original_decode(*args, **kwargs)
            torch.cuda.synchronize()
            timing.setdefault('vae_decode', []).append(time.perf_counter() - started)
            return result

        pipe.vae.decode = decode

        def weight_bytes(module):
            return sum(p.numel() * p.element_size() for p in module.parameters())

        def measure(model_name):
            report.setdefault('models', {})[model_name] = {
                'transformer_parameters': sum(p.numel() for p in pipe.transformer.parameters()),
                'transformer_bytes': weight_bytes(pipe.transformer), 'text_encoder_bytes': weight_bytes(pipe.text_encoder),
                'vae_bytes': weight_bytes(pipe.vae), 'blocks': len(pipe.transformer.transformer_blocks)}
            for steps in request['steps']:
                name = f'{model_name}_{steps}'
                for row in request['warmup_rows']:
                    with torch.inference_mode():
                        pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=steps,
                             true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil')
                runs = []
                for repeat in range(request['repeats']):
                    for row in request['rows']:
                        timing.clear()
                        torch.cuda.reset_peak_memory_stats()
                        torch.cuda.synchronize()
                        started = time.perf_counter()
                        with torch.inference_mode():
                            image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=steps,
                                         true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                         use_kv_cache=True, output_type='pil').images[0]
                        torch.cuda.synchronize()
                        total = time.perf_counter() - started
                        calls = timing['transformer']
                        runs.append({'source_id': row['source_id'], 'repeat': repeat, 'total_seconds': total,
                                     'transformer_seconds': sum(calls), 'transformer_calls': len(calls),
                                     'first_step_seconds': calls[0], 'later_step_seconds': statistics.median(calls[1:]) if len(calls) > 1 else None,
                                     'text_encoder_seconds': sum(timing.get('text_encoder', [])), 'vae_decode_seconds': sum(timing['vae_decode']),
                                     'peak_memory_bytes': torch.cuda.max_memory_allocated()})
                        if repeat == 0:
                            path = directory / (row['source_id'] + '-' + name + '.png')
                            image.save(path, format='PNG')
                            outputs.setdefault(name, {})[row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size}
                report.setdefault('runs', {})[name] = runs
                report['calls'][name] = len(runs)
                (directory / 'progress.json').write_text(json.dumps(report, indent=2))
                results_volume.commit()

        model.eval()
        measure('teacher')
        # Version 1: blocks 5 and 20 replaced by their attention bridges (unhealed).
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=True)
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()
        measure('v1')
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_bench():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    healing = load(next((root / 'data/phase8/healing').glob('*/request.json')).relative_to(root))['request']
    general = [r for r in base['rows'] if r['subdimension'] not in ('text_rendering', 'posters')]
    square = lambda r: dict(r, width=1024, height=1024)
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase9-benchmark-v1', worker_source=SOURCES['benchmark_worker.py'], execution_account='new', candidate_only=True,
        conditioned_bridge_source=healing['conditioned_bridge_source'], bridges=healing['bridges'],
        steps=[40, 8, 4], repeats=3, rows=[square(r) for r in general[:5]], warmup_rows=[square(general[5])],
        scope='Speed and memory, batch 1 at 1024x1024 on one H100: teacher (32 blocks) and version 1 (30 blocks), 40, 8 and 4 steps')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase9/benchmark' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase9/benchmark/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase9-benchmark')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'benchmark_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key


def benchmark_table(report):
    import statistics
    head = ('| Model | Steps | Image (s) | Transformer (s) | Later step (ms) | Text encoder (s) | VAE decode (s) | Peak memory (GB) |\n'
            '|---|---|---|---|---|---|---|---|\n')
    lines = []
    for name, runs in report['runs'].items():
        model_name, steps = name.rsplit('_', 1)
        median = lambda k: statistics.median(r[k] for r in runs)
        lines.append(f"| {model_name} | {steps} | {median('total_seconds'):.2f} | {median('transformer_seconds'):.2f} | "
                     f"{1000 * median('later_step_seconds'):.0f} | {median('text_encoder_seconds'):.2f} | {median('vae_decode_seconds'):.2f} | "
                     f"{median('peak_memory_bytes') / 1e9:.1f} |")
    display(Markdown(head + chr(10).join(lines)))
    for model_name, m in report['models'].items():
        print(model_name, m['blocks'], 'blocks;', f"transformer {m['transformer_parameters'] / 1e9:.3f}B parameters, "
              f"{m['transformer_bytes'] / 1e9:.2f} GB; text encoder {m['text_encoder_bytes'] / 1e9:.2f} GB; VAE {m['vae_bytes'] / 1e9:.2f} GB")

LAUNCH_ARGS = ['--dry-run']
RUN_BENCH = 'data/phase9/benchmark/' + main_bench()
LAUNCH_ARGS = []
summary_bench = reuse_or_launch(RUN_BENCH, main_bench)
if summary_bench and summary_bench.get('status') == 'passed':
    benchmark_table(summary_bench)